In [1]:
import os
import socket
from pathlib import Path
from urllib.parse import quote_plus

import polars as pl
import pandas as pd
import plotly.express as px
from dotenv import load_dotenv
import ibm_db
import ibm_db_dbi
import ibm_db_sa
from sqlalchemy import create_engine, text
from sqlalchemy.dialects import registry as _sa_registry
_sa_registry.register("db2", "ibm_db_sa.ibm_db", "DB2Dialect_ibm_db")
_sa_registry.register("db2.ibm_db", "ibm_db_sa.ibm_db", "DB2Dialect_ibm_db")

load_dotenv()
print(f"Polars version: {pl.__version__}")


Polars version: 1.41.2


In [2]:
DB_HOST = os.getenv("DB_HOST", "52.211.123.34")
DB_PORT = int(os.getenv("DB_PORT", "25010"))
DB_NAME = os.getenv("DB_NAME", "ATTPLANE")
DB_USERNAME = os.getenv("DB_USERNAME", "attgrp8")   # ← change to your group number
DB_PASSWORD = os.getenv("DB_PASSWORD", "bigdata")

print({"host": DB_HOST, "port": DB_PORT, "database": DB_NAME, "username": DB_USERNAME, "password": "***"})

{'host': '52.211.123.34', 'port': 25010, 'database': 'ATTPLANE', 'username': 'attgrp8', 'password': '***'}


In [3]:
def tcp_check(host: str = DB_HOST, port: int = DB_PORT, timeout: float = 8.0) -> bool:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.settimeout(timeout)
        sock.connect((host, port))
    return True

try:
    tcp_check()
    print(f"TCP connection succeeded: {DB_HOST}:{DB_PORT} is reachable")
except Exception as exc:
    print(f"TCP connection failed: {type(exc).__name__}: {exc}")

TCP connection succeeded: 52.211.123.34:25010 is reachable


In [4]:
def _make_raw_connection():
    conn_str = (
        f"HOSTNAME={DB_HOST};PORT={DB_PORT};DATABASE={DB_NAME};"
        f"PROTOCOL=TCPIP;UID={DB_USERNAME};PWD={DB_PASSWORD};"
        f"AUTHENTICATION=SERVER;CURRENTSCHEMA={DB_USERNAME.upper()};"
    )
    return ibm_db_dbi.Connection(ibm_db.connect(conn_str, "", ""))

def make_db2_engine():
    return create_engine("ibm_db_sa://", creator=_make_raw_connection)

engine = make_db2_engine()
engine


Engine(ibm_db_sa://)

In [5]:
def test_db_connection(engine) -> bool:
    with engine.connect() as conn:
        result = conn.execute(text("SELECT 1 AS ok FROM SYSIBM.SYSDUMMY1"))
        row = result.fetchone()
    return row is not None and row[0] == 1

try:
    assert test_db_connection(engine)
    print("DB2 connection successful")
except Exception as exc:
    print("DB2 connection failed")
    print(f"{type(exc).__name__}: {exc}")
    print("\nChecklist:")
    print("1. Did the TCP check succeed?")
    print("2. Are ibm_db and ibm_db_sa installed in this notebook kernel?")
    print("3. Are DB_USERNAME and DB_PASSWORD correct for your group?")
    print("4. Is DB_NAME exactly ATTPLANE?")

DB2 connection successful
